# Plotting

We present some examples for how to use the basic plotting functions in the python interface for GVEC. Note that
these are not intended for advanced plotting, but for quickly visualising some basic properties.
We will use the same elliptic stellarator as in the [elliptic stellarator](</tutorials/notebooks/020_stellarator.ipynb>) for the plotting examples.

In [ ]:
import gvec

In [ ]:
params = {
    "ProjectName": "test_plotting",
    "which_hmap": 1,
    "PhiEdge": 1.0,
    "iota": {"type": "polynomial", "coefs": [0.625, 0.35]},
    "pres": {"type": "polynomial", "coefs": [1.0, -1.0], "scale": 1000.0},
    "nfp": 3,
    "X1_b_cos": {(0, 0): 3.0, (1, 0): 1.0, (1, 1): 0.4},
    "X2_b_sin": {(1, 0): 1.0, (1, 1): -0.4, (0, 1): -0.25},
    "init_average_axis": True,
    "sgrid_nElems": 2,
    "X1_mn_max": [3, 3],
    "X2_mn_max": [3, 3],
    "LA_mn_max": [3, 3],
    "X1X2_deg": 5,
    "LA_deg": 5,
    "totalIter": 1000,
    "minimize_tol": 1.0e-6,
}


try:
    state = gvec.find_state()
except ValueError:
    run = gvec.run(params)
    state = run.state


Note that all 1D and 2D plotting functions return the `matplotlib.figure.Figure` object, so you may edit the output if you wish.

# 1D plots
Two are two convience functions for plotting 1D properties.
The first is to plot radial profiles, i.e. scalar functions of $\rho$. Note that only scalar quantities can be evaluated directly.

In [ ]:
fig, ax = state.plot_radial_profile()

Properties along the magnetic axis can be plotted with `plot_on_axis`. Since some derived quantities cannot be evaluated on axis,
this will use quadratic extrapolation to generate the values for plotting. Also note that although we have specified the subplot grid here, this is not required unless
you want a specific shape.

In [ ]:
fig, ax = state.plot_on_axis(quantities=["mod_B", "X1"], subplot_grid=[2, 1])

# 2D plotting

## Poloidal slice plots

For plotting poloidal slices we can lock or unlock the $X^1$ and $X^2$ values on the $x$ and $y$ axes by specifying `share_axis=True/False` (by default this is `True`). By default we also plot contours of fixed $\rho$ and $\vartheta^\star$ values (PEST coordinates).

In [ ]:
fig, ax = state.plot_poloidal_plane(zeta=4)

## Flux surface plots

For plotting values on specific flux surfaces we use `plot_on_flux_surface`, note that multiple flux surfaces can be shown at once by specifying ether a list of flux surfaces labels, of a `int` of the number of surfaces. By default the last closed flux surface (GVEC boundary) is plotted. 

Also note that by default the plot is in Boozer coordinates. This can be changed to either PEST or normal $(\vartheta,\zeta)$ coordinates by setting `sfl="pest"` or `sfl=None`.

In [ ]:
fig, ax = state.plot_on_flux_surface(rho=[0.3, 0.6])

Note that we can also evaluate different values on the same flux surface by specifying the `quantities` rather than multiple `rho` values (currently we cannot do both). Note that the quantities should be scalar values, as with the 1D plots. In the plot below we also display filled contours, and show the quantities in regular $(\vartheta,\zeta)$ coordinates rather than straight-field-line coordinates.

In [ ]:
fig, ax = state.plot_on_flux_surface(
    quantities=["mod_B", "Jac"], subplot_grid=[2, 1], sfl=None, style="filled-contour"
)

# 3D plotting

For 3D plotting we use [plotly](https://plotly.com/python/) as a backend.
To plot the boundary we only need the state file and the resolution of the plot. Note that we can also specify the quantity we want to plot with
the `quantities` keyword, by default $\|B\|$ will be plotted on the boundary.

3D plots return the `plotly.graph_objs._figure.Figure` object.

Note that you may need a version of `plotly<6.0` in order to display the plots in a jupyter notebook.

In [ ]:
fig = state.plot_3d_surface()
fig.show();

The `plot_boundary` function is implemented as a convience function sets `rho=1.0` as input to the function `plot_3d_surface`.
If for some reason _plotly_ does not `show`, there is an optional input, `to_file` (default `None`),
which can be set to a string to write the plot to a file in your current working directory.

In [ ]:
fig = state.plot_3d_surface("L_gradB", rho=0.5, ntheta=31, nzeta=41)
fig.show();